# Essentia DEAM MusiCNN — Valence Key Correction Benchmark

Evaluates the **deam-musicnn** Essentia head with the key-mode valence correction applied.
This combines the strongest cross-dataset Essentia backbone (`deam-msd-musicnn-2.pb`)
with the valence calibration from `feat/valence_key_correction`.

**Correction formula:** `valence += α × (is_major − 0.5) × key_strength`  
where `α = 0.13` (calibrated on EmoMusic: mean valence major ≈ 5.8/9 vs minor ≈ 4.6/9)
and `key_strength` is the HPCP-based KeyExtractor confidence ∈ [0, 1].

| Model | Backbone | Head | Correction |
|-------|----------|------|------------|
| **This notebook** | MusiCNN | DEAM | Key-mode valence |
| `01_essentia_benchmark` | MusiCNN | EmoMusic | None |
| `01c_essentia_valence_key_correction` | MusiCNN | EmoMusic | Key-mode valence |

> **NOTE:** essentia-tensorflow is CPU-only — selecting a GPU runtime does not help here.

### Sections
0. Environment setup
1. Shared evaluation utilities
2. Essentia model files
3. Predictor (DEAM head + key-mode valence correction)
4. Datasets
5. Evaluation
6. Qualitative spot-checks
7. Profiling
8. Summary

## 0. Environment Setup

In [ ]:
!pip install essentia-tensorflow yt-dlp librosa matplotlib pandas scipy tqdm gdown -q
print("Setup complete.")

## 1. Shared Evaluation Utilities

In [ ]:
import os, sys
from pathlib import Path

# REPO_BRANCH: update to "main" after the PR is merged
REPO_BRANCH = "feat/mood-model-benchmark"
REPO_NAME   = "Soundtrack-Mood-Manager"

_cwd = Path.cwd()
if (_cwd / "eval_datasets.py").exists():
    _eval_dir = _cwd
else:
    _repo_root = _cwd / REPO_NAME
    if not (_repo_root / "evaluation").exists():
        !git clone --depth 1 --branch {REPO_BRANCH} \
            https://github.com/francescovidaich964/{REPO_NAME}.git
    _eval_dir = _repo_root / "evaluation"

_eval_dir = _eval_dir.resolve()
if str(_eval_dir) not in sys.path:
    sys.path.insert(0, str(_eval_dir))

from eval_datasets import setup_deam, setup_emomusic, setup_pmemo, setup_merge
from metrics import compute_metrics, print_metrics
from visualization import plot_scatter, cross_dataset_comparison
from spot_checks import SPOT_CHECKS, download_spot_checks, run_evaluation, profile_predictor

import math, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR      = Path("data")
MODELS_DIR    = Path("models")
SPOTCHECK_DIR = Path("spotchecks")
for d in [DATA_DIR, MODELS_DIR, SPOTCHECK_DIR]:
    d.mkdir(exist_ok=True)

MODEL_TAG = "deam-musicnn-valence-corrected"
print(f"Imports complete. (eval_dir={_eval_dir})")

## 2. Download Essentia Model Files

MusiCNN feature extractor (shared with production) + DEAM regression head.

In [ ]:
MODEL_URLS = {
    "msd-musicnn-1.pb":
        "https://essentia.upf.edu/models/feature-extractors/musicnn/msd-musicnn-1.pb",
    "deam-msd-musicnn-2.pb":
        "https://essentia.upf.edu/models/classification-heads/deam/deam-msd-musicnn-2.pb",
}

for name, url in MODEL_URLS.items():
    dest = MODELS_DIR / name
    if dest.exists():
        print(f"  ✓ {name} ({dest.stat().st_size/1e6:.1f} MB, cached)")
        continue
    print(f"Downloading {name}...", end=" ", flush=True)
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(65536):
            f.write(chunk)
    print(f"{dest.stat().st_size/1e6:.1f} MB")

## 3. Predictor

In [ ]:
# Valence correction alpha: calibrated on EmoMusic (major vs minor gap ≈ 0.13)
_ALPHA = 0.13


class EssentiaPredictor:
    """DEAM MusiCNN head with key-mode valence correction.

    Uses deam-msd-musicnn-2.pb as the regression head instead of the production
    emomusic head. The DEAM head showed the best cross-dataset valence and arousal
    performance in the essentia models comparison (01b).

    The key-mode correction is the same as in 01c — alpha=0.13 calibrated on EmoMusic:
    full-confidence major shifts valence +0.065; full-confidence minor shifts −0.065.

    NOTE: essentia-tensorflow is CPU-only regardless of runtime type.
    """

    def __init__(self, models_dir: Path, batch_size: int = 256):
        import essentia
        essentia.log.warningActive = False
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            from essentia.standard import (
                MonoLoader, TensorflowPredictMusiCNN, TensorflowPredict2D, KeyExtractor,
            )
        self._MonoLoader = MonoLoader
        self._musicnn = TensorflowPredictMusiCNN(
            graphFilename=str(models_dir / "msd-musicnn-1.pb"),
            output="model/dense/BiasAdd",
            batchSize=batch_size,
        )
        self._head = TensorflowPredict2D(
            graphFilename=str(models_dir / "deam-msd-musicnn-2.pb"),
            output="model/Identity",
            batchSize=batch_size,
        )
        self._key_extractor = KeyExtractor(sampleRate=16000)
        print(f"EssentiaPredictor (deam-musicnn + valence correction) ready (CPU, batchSize={batch_size}).")

    def predict(self, audio_path) -> dict | None:
        """Returns {'valence': float, 'arousal': float} in [0, 1], or None on failure."""
        try:
            audio = self._MonoLoader(filename=str(audio_path), sampleRate=16000)()

            try:
                _key, scale, key_strength = self._key_extractor(audio)
                is_major = 1.0 if scale == "major" else 0.0
            except Exception:
                is_major, key_strength = 0.5, 0.0  # neutral: no correction

            embeddings = self._musicnn(audio)
            if embeddings.shape[0] == 0:
                return None
            preds = self._head(embeddings)
            if preds.shape[0] == 0:
                return None
            mean = preds.mean(axis=0)
            valence = float(np.clip((mean[0] - 1.0) / 8.0, 0.0, 1.0))
            arousal = float(np.clip((mean[1] - 1.0) / 8.0, 0.0, 1.0))

            valence = float(np.clip(valence + _ALPHA * (is_major - 0.5) * key_strength, 0.0, 1.0))

            if not (math.isfinite(valence) and math.isfinite(arousal)):
                return None
            return {"valence": valence, "arousal": arousal}
        except Exception:
            return None


predictor = EssentiaPredictor(MODELS_DIR)

## 4. Datasets

In [ ]:
df_deam,     deam_id, deam_val, deam_aro = setup_deam(DATA_DIR,  download_audio=True)
df_emomusic, em_id,   em_val,   em_aro   = setup_emomusic(DATA_DIR)
df_pmemo,    pm_id,   pm_val,   pm_aro   = setup_pmemo(DATA_DIR)
df_merge,    mg_id,   mg_val,   mg_aro   = setup_merge(DATA_DIR, download_audio=True)

## 5. Evaluation

essentia-tensorflow is CPU-only — selecting a GPU runtime does not help.
At ~13 s/track on CPU: DEAM ~6.5 h, PMEmo ~2.8 h, MERGE ~11.7 h.
Set `MAX_TRACKS` to a small number first to verify the predictor works.

In [ ]:
_deam_audio   = next(iter(sorted((DATA_DIR / "deam").rglob("MEMD_audio"))), None)
DEAM_AUDIO    = _deam_audio if _deam_audio else DATA_DIR / "deam" / "MEMD_audio"
if _deam_audio:
    print(f"DEAM audio  : {DEAM_AUDIO}")

EMOMUSIC_AUDIO = DATA_DIR / "emomusic" / "clips"

_pmemo_chorus = next(iter(sorted((DATA_DIR / "pmemo").rglob("chorus"))), None)
PMEMO_AUDIO   = _pmemo_chorus if _pmemo_chorus else DATA_DIR / "pmemo" / "chorus"
if _pmemo_chorus:
    print(f"PMEmo audio : {PMEMO_AUDIO}")

_merge_root   = next(iter(sorted((DATA_DIR / "merge").rglob("MERGE_Audio_Balanced"))), None)
MERGE_AUDIO   = _merge_root if _merge_root else DATA_DIR / "merge" / "MERGE_Audio_Balanced"
if _merge_root:
    print(f"MERGE audio : {MERGE_AUDIO}")

MAX_TRACKS = None

DATASETS = [
    ("DEAM",     df_deam,     DEAM_AUDIO,     deam_id, deam_val, deam_aro),
    ("EmoMusic", df_emomusic, EMOMUSIC_AUDIO, em_id,   em_val,   em_aro),
    ("PMEmo",    df_pmemo,    PMEMO_AUDIO,    pm_id,   pm_val,   pm_aro),
    ("MERGE",    df_merge,    MERGE_AUDIO,    mg_id,   mg_val,   mg_aro),
]

all_results = {}
for ds_name, df_a, audio_dir, id_col, val_col, aro_col in DATASETS:
    if df_a is None or not audio_dir.exists():
        print(f"Skipping {ds_name} — audio not found at {audio_dir}")
        continue
    all_results[ds_name] = run_evaluation(
        ds_name, MODEL_TAG, predictor.predict,
        audio_dir, df_a, id_col, val_col, aro_col, MAX_TRACKS,
    )

In [ ]:
for ds_name, df in all_results.items():
    print_metrics(df, ds_name)
    plot_scatter(df, f"DEAM MusiCNN + correction — {ds_name}")

In [ ]:
if all_results:
    combined = pd.concat(all_results.values())
    out = f"{MODEL_TAG}_results.csv"
    combined.to_csv(out, index=False)
    print(f"Saved: {out}  ({len(combined)} rows)")
else:
    print("No results to save — check that audio directories exist.")

## 6. Qualitative Spot-checks

In [ ]:
download_spot_checks(SPOTCHECK_DIR)

In [ ]:
spot_rows = []
for track in SPOT_CHECKS:
    audio = SPOTCHECK_DIR / f"{track['title']}.mp3"
    if not audio.exists():
        print(f"  ✗ {track['title']} — not found, skipping")
        continue
    pred = predictor.predict(audio)
    spot_rows.append({
        "title":       track["title"].replace("_", " "),
        "exp_valence": track["exp_valence"],
        "exp_arousal": track["exp_arousal"],
        "valence":     pred["valence"] if pred else float("nan"),
        "arousal":     pred["arousal"] if pred else float("nan"),
    })
    name = track["title"].replace("_", " ").title()
    print(f"\n{name}")
    print(f"  Expected:  v={track['exp_valence']:.2f}  a={track['exp_arousal']:.2f}")
    if pred:
        print(f"  Predicted: v={pred['valence']:.2f}  a={pred['arousal']:.2f}")
    else:
        print("  Predicted: FAILED")

spot_df = pd.DataFrame(spot_rows)
print(f"\nCompleted {len(spot_df)} spot-checks.")

In [ ]:
if not spot_df.empty:
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(spot_df)))
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.5, color="#ccc", lw=0.8, ls="--"); ax.axvline(0.5, color="#ccc", lw=0.8, ls="--")
    ax.text(0.02, 0.98, "sad/energetic",   transform=ax.transAxes, va="top",    fontsize=8, color="#999")
    ax.text(0.98, 0.98, "happy/energetic",  transform=ax.transAxes, va="top",    ha="right", fontsize=8, color="#999")
    ax.text(0.02, 0.02, "sad/calm",         transform=ax.transAxes, va="bottom", fontsize=8, color="#999")
    ax.text(0.98, 0.02, "happy/calm",       transform=ax.transAxes, va="bottom", ha="right", fontsize=8, color="#999")
    for i, row in spot_df.iterrows():
        c = colors[i]
        ax.scatter(row["exp_valence"], row["exp_arousal"], marker="*", s=220, color=c, zorder=4)
        if not pd.isna(row["valence"]):
            ax.scatter(row["valence"], row["arousal"], marker="o", s=70,
                       color=c, edgecolors="black", lw=0.5, zorder=4)
            ax.annotate("", xy=(row["valence"], row["arousal"]),
                        xytext=(row["exp_valence"], row["exp_arousal"]),
                        arrowprops=dict(arrowstyle="->", color=c, lw=1.0))
        ax.annotate(row["title"], xy=(row["exp_valence"], row["exp_arousal"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=7, color=c)
    ax.set_xlabel("Valence (sad → happy)")
    ax.set_ylabel("Arousal (calm → energetic)")
    ax.set_title("DEAM MusiCNN + correction — Spot-checks\n★ = expected  ● = predicted")
    plt.tight_layout()
    plt.savefig(f"{MODEL_TAG}_spotchecks.png", dpi=120, bbox_inches="tight")
    plt.show()

## 7. Runtime & Memory Profiling

In [ ]:
test_audio = next(
    (SPOTCHECK_DIR / f"{t['title']}.mp3" for t in SPOT_CHECKS
     if (SPOTCHECK_DIR / f"{t['title']}.mp3").exists()),
    None,
)

if test_audio is None:
    print("No audio for profiling — run Section 6 first.")
else:
    prof = profile_predictor(predictor.predict, test_audio, n=5)
    print(f"DEAM MusiCNN + correction — profiling on {test_audio.name}:")
    print(f"  Mean: {prof['mean_s']:.2f} s/track")
    print(f"  Std:  {prof['std_s']:.3f} s")
    print(f"  Peak RAM: {prof['peak_mb']:.1f} MB")

## 8. Summary

In [ ]:
print("=" * 60)
print("DEAM MUSICNN + VALENCE CORRECTION — BENCHMARK SUMMARY")
print("=" * 60)

print("\n── Dataset metrics ──")
if all_results:
    combined = pd.concat(all_results.values())
    print_metrics(combined, "all datasets")
else:
    print("  (no datasets evaluated)")

print("\n── Spot-checks ──")
if not spot_df.empty:
    print(spot_df[["title", "exp_valence", "exp_arousal", "valence", "arousal"]]
          .to_string(index=False, float_format="{:.2f}".format))
else:
    print("  (none run)")

print("\n── Runtime ──")
if "prof" in dir():
    print(f"  {prof['mean_s']:.2f} s/track  (peak RAM {prof['peak_mb']:.1f} MB)")
else:
    print("  (run Section 7 to profile)")

print("\n" + "=" * 60)